# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con google/flan-t5-base

Model page: https://huggingface.co/google/flan-t5-base

# No keywords

In [4]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships No Keywords"
hf_model_repo = "google/flan-t5-base"
model_name = "flanT5"  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(hf_model_repo)
model =AutoModelForSeq2SeqLM.from_pretrained(hf_model_repo).to(device)
model.eval()


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [5]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2212, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 2212 (delta 231), reused 201 (delta 141), pack-reused 1920 (from 1)
Receiving objects: 100% (2212/2212), 73.65 MiB | 12.10 MiB/s, done.
Resolving deltas: 100% (1870/1870), done.
Updating files: 100% (1019/1019), done.


In [6]:
def preprocess_text(s):
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def build_prompt(arg1, arg2):
    return (
        "Classify the relationship between the following two arguments.\n"
        "Respond with exactly one label from: Support, Attack, Rephrase, No Relationship.\n\n"

        f"Argument 1: {arg1}\n"
        f"Argument 2: {arg2}\n\n"
        "Label:"
    )

def normalize_label(text):
    t = (text or "").strip().lower()
    if "support" == t or t.startswith("support"):
        return "Support"
    if "attack" == t or t.startswith("attack") or "conflict" in t:
        return "Attack"
    if "rephrase" == t or "paraphrase" in t or t.startswith("reph"):
        return "Rephrase"
    if "no relationship" == t or "no relation" in t or "none" == t:
        return "No Relationship"
    # fallbacks for short/partial tokens
    if t in {"support", "attack", "rephrase"}:
        return t.capitalize()
    return "No Relationship"

def compute_max_length_for_prompts(input_dir, prefix_substring, tokenizer, safety_limit=None):   
    if safety_limit is None:
        safety_limit = min(getattr(tokenizer, "model_max_length", 512) or 512, 512)

    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        df = pd.read_csv(path)
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = preprocess_text(str(a) if pd.notna(a) else "")
            b = preprocess_text(str(b) if pd.notna(b) else "")
            prompt = build_prompt(a, b)
            ids = tokenizer(prompt, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(ids))
    return min(max_len if max_len > 0 else 128, safety_limit)


def predict_labels_flan(prompts, max_length):
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_length
    ).to(device)

    outputs = model.generate(
        **enc,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,              
        num_beams=1,
    )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    labels = [normalize_label(x) for x in decoded]
    labels = [l if l in VALID_OUT else "No Relationship" for l in labels]
    return labels

def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def classify_relationships_flan(input_dir, prefix_substring):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    # compute a safe prompt max length once per run
    MAX_LENGTH = compute_max_length_for_prompts(input_dir, prefix_substring, tokenizer)
    print(f"Using MAX_LENGTH = {MAX_LENGTH}")

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]

            prompts = []
            a1_list = []
            a2_list = []
            for a1, a2 in zip(chunk["SDGarg1"], chunk["SDGarg2"]):
                a1 = preprocess_text(str(a1) if pd.notna(a1) else "")
                a2 = preprocess_text(str(a2) if pd.notna(a2) else "")
                prompts.append(build_prompt(a1, a2))
                a1_list.append(a1)
                a2_list.append(a2)

            try:
                labels = predict_labels_flan(prompts, max_length=MAX_LENGTH)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(prompts)

            df.loc[chunk.index, rel_col] = labels

            for a1, a2, lab in zip(a1_list, a2_list, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Sample predictions (first 5):\n")
                    for a1, a2, lab in first_examples:
                        print(f"- Arg1: {a1[:]}\n-Arg2: {a2[:]}\nLabel: {lab}\n\n")
                    first_examples.append((a1, a2, lab))                                        

            total_done += len(prompts)
            if total_done % 100 < BATCH_SIZE: 
                print(f"  Progress: {total_done}/{n} relations classified...")

        # handle missing args (safety)
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"

        # Ensure valid set
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
        return df

## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 213

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: according to major international studies, few teenagers can differentiate between a fact and an opinion.
Label: No Relationship


- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: as the world’s nations prepare to meet in september to review the progress the world has made so far towards achieving the sdgs, at the midpoint of the 2030 agenda, sdsn emphasizes six areas for immediate action.
Label: Support


- Arg1: at their core, the sdgs are an investment agenda: 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5
359,"For these reasons, we end our message with two...","Building on the past ten years of work, includ...",0_14,0_31,NaN,No Relationship,Support
333,The continuing efforts of the SDSN is a testam...,Achieving the SDGs requires global cooperation...,0_13,0_22,NaN,Support,Support
280,The “global financial architecture” (GFA) refe...,It is argued since 2017 that a combination of ...,0_10,0_26,NaN,No Relationship,Support
742,"The interconnected environmental, social, and ...",1. Greatly increase funding to national and su...,10_2,10_4,NaN,Support,Support
494,Rich European countries top the overall SDG In...,"Building on the past ten years of work, includ...",0_29,0_31,NaN,No Relationship,Support


rel_flanT5
Support            626
No Relationship    424
Attack               8
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 228

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: only limited progress is being made on the environmental and biodiversity goals, including sdg 12 (responsible consumption and production), sdg 13 (climate action), sdg 14 (life below water), and sdg 15 (life on land)
Label: No Relationship


- Arg1: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
-Arg2: and global cooperation has ebbed as geopolitical tensions have risen.
Label: No Relationship


- Arg1: at their core, the sdgs are an investment agenda: it is critical that un

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5
6525,The SDG Index acknowledges Bhutan’s recent pro...,"At this midpoint of the 2030 Agenda, all count...",3_10,16_9,NaN,Support,Support
11475,The world is also seriously off track to meet ...,SDG 17 (Partnerships for the Goals) calls for ...,13_2,17_11,NaN,No Relationship,Support
6457,Universal health access and coverage,The disastrous war in Ukraine has further dest...,3_4,16_1,NaN,No Relationship,No Relationship
1372,Official high-level speeches and the preparati...,ensure universal access to modern energy sources,0_18,7_8,NaN,No Relationship,Support
11784,47 countries have committed to submitting a VN...,To make sure that existing financial resources...,16_7,17_7,NaN,No Relationship,Support


rel_flanT5
No Relationship    5987
Support            5665
Attack              170
Name: count, dtype: int64

#### Gemma3 27B extraction

In [6]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 275

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: despite this alarming development, the sdgs are still achievable.
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the sdgs the world must both alter its current investment patterns and increase the overall volume of investments.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: national governments must 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5
3733,And global cooperation has ebbed as geopolitic...,The urgent objective of the SDG Stimulus is to...,17_9,17_20,NaN,No Relationship,Support
3983,Increased funding from the Multilateral Develo...,These fora are critical to encourage internati...,17_19,17_25,NaN,Support,Support
979,"Despite this ominous news, the SDGs are still ...",SDSN has recommended six inter-related long-te...,0_24,0_32,NaN,No Relationship,Support
475,"All UN Member States should present, at regula...",All UN Member States should recommit to peacef...,0_10,0_11,NaN,Support,Support
2720,"protecting biodiversity, sustainably managing ...","The Climate Action Tracker, an independent sci...",13_13,13_22,NaN,Attack,Support


rel_flanT5
Support            2920
No Relationship    1168
Attack              100
Name: count, dtype: int64

In [7]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 336

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at the global level, averaging across countries, not a single sdg is currently projected to be met by 2030, with the poorest countries struggling the most.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: as called for by united nations secretary-general antónio guterres, the sdg stimulus plan has five main components:
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (lics) and lower-middle-income countries (lmics), to carry out needed sdg actions;
Label: No Relati

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5
8389,All UN Member States should adopt long-term su...,The world is also seriously off track to meet ...,0_9,13_1,NaN,Attack,No Relationship
19984,poor countries need help to combat poverty.,The SDSN and its global network will double-do...,1_19,17_7,NaN,Support,Support
11967,Long-term investment plans are essential for n...,All UN Member States should recommit to peacef...,0_31,16_0,NaN,Support,Support
9019,Achieving the SDGs will require a transformati...,We firmly believe that international cooperati...,0_29,13_11,NaN,Support,Support
15031,Although all governments are in principle comm...,Local governments have the front-line responsi...,1_6,3_2,NaN,Support,Support


rel_flanT5
Support            29757
No Relationship    19072
Attack              1268
Name: count, dtype: int64

#### Gemma3 4B extraction

In [8]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 318

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: none of their objectives are beyond our reach.
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the sdgs are still achievable.
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the sdgs the world must both alter its current investment patterns and increase the overall volume of investments.
Label: Support


- Arg1: at the midpoint of the 2030 agen

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
22058,Alignment of private business investment flows...,The international financial architecture is fa...,17_5,17_11,NaN,No Relationship
21838,Revise liquidity structures for LICs and LMICs...,A reform of current institutional frameworks a...,17_0,17_6,NaN,Support
10105,SDSN is closely following and supporting the E...,The UN Secretary-General António Guterres has ...,0_79,0_152,NaN,No Relationship
2939,UN Member States must endorse a deep and overd...,no single G20 country has adopted a sufficient...,0_18,0_123,NaN,No Relationship
14466,SDS N’s Europe SDR emphasizes the importance o...,HICs are able to mobilize vast financial resou...,1_15,1_21,NaN,No Relationship


rel_flanT5
Support            14340
No Relationship     8389
Attack               185
Name: count, dtype: int64

In [9]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 325

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the world is off track, but that is all the more reason to double down on the sdgs.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: revise liquidity structures for lics and lmics, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises.
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: create ambitious, internationa

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
204646,globalized trade rules for ‘cleantech’ could a...,A society cannot function peacefully if there ...,10_17,16_8,NaN,No Relationship
97258,Overhauling global governance mechanisms and t...,"Sustainable ecosystems, sustainable agricultur...",1_22,7_1,NaN,Support
40937,The ESDR has inspired strong and meaningful po...,Align private business investment flows with t...,0_149,8_2,NaN,Support
53658,Further analyses will be needed to capture pol...,there is a risk that the gap in SDG outcomes b...,0_128,10_8,NaN,No Relationship
666,Further investment is needed in statistical ca...,The HICs’ somewhat better performance on pilla...,0_13,1_42,NaN,Support


rel_flanT5
Support            129410
No Relationship     91509
Attack               5032
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [10]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 318

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: since the outbreak of the pandemic in 2020 and other simultaneous crises, sdg progress has stalled globally.
Label: Attack


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the world is off track, but that is all the more reason to double down on the sdgs.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the sdgs the world mu

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5
4940,"Above all, the SDGs represent an investment ag...",The dashboards highlight persisting gaps betwe...,0_65,0_66,NaN,Support,No Relationship
10880,"First, implementation is largely left to the n...",Although the World Bank’s Statistical Performa...,17_19,17_65,NaN,Support,Support
2401,"In her 2022 report on the SDGs, E. Tendayi Ach...",Investing in the SDGs,0_25,0_27,NaN,No Relationship,No Relationship
11117,SDSN puts a great emphasis on long-term nation...,"The great seas, such as the Mediterranean Sea ...",17_25,17_35,NaN,No Relationship,Support
1564,UN Member States must endorse a deep and overd...,"Above all, the SDGs represent an investment ag...",0_15,0_65,NaN,Support,Support


rel_flanT5
Support            8623
No Relationship    3175
Attack              213
Name: count, dtype: int64

In [11]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 367

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at the global level, averaging across countries, not a single sdg is currently projected to be met by 2030, with the poorest countries struggling the most.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the grim reality is that at the midpoint of the 2030 agenda, the sdgs are far off track.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries.
Label: Attack


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off tra

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5
10638,Science-based instruments are needed at all le...,Private capital markets continue to direct lar...,0_105,7_4,NaN,Support
19582,Dire shortfalls in meeting the SDGs The SDGs a...,"Yet the SDG Dashboards rate rich countries, in...",0_21,13_48,NaN,No Relationship
21679,Multiple and overlapping health and geopolitic...,Both the 2030 Agenda and the Paris Climate Agr...,0_61,13_25,NaN,No Relationship
14921,Building on earlier work conducted by the SDSN...,Part of this progress might be due to investme...,0_73,10_18,NaN,Support
73425,National government must also work with subnat...,Private capital markets continue to direct lar...,9_7,13_26,NaN,No Relationship


rel_flanT5
Support            60529
No Relationship    37937
Attack              3049
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [7]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 437

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: despite this alarming development, the sdgs are still achievable. none of their objectives are beyond our reach.
Label: No Relationship


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the world is off track, but that is all the more reason to double down on the sdgs.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at their core, the sdgs are an investment agenda: it is critical that un member states adopt and implement the sdg stimulus and support a comprehensive reform of the global financial architecture.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the s

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL,rel_llama,rel_flanT5
13087,1. The SDSN Framework release a massive new lo...,"EU-wide financial resources, notably the EU Re...",13_16,13_23,NaN,No Relationship,NaN,No Relationship
6000,The United States has so far shown very little...,The Index also helps shed light on certain key...,0_51,0_136,NaN,No Relationship,NaN,Support
4056,An estimated 1.8 billion people depend on drin...,The IMF should build its national reviews (Art...,0_32,0_73,NaN,No Relationship,NaN,Support
12913,rising sea levels (including the growing possi...,"Compared with other SDG monitoring reports, ho...",13_10,13_44,NaN,Support,NaN,No Relationship
3554,"According to the annual SDG Index, global achi...",Disadvantaged communities have lower access to...,0_27,0_126,NaN,No Relationship,NaN,No Relationship


rel_flanT5
Support            11365
No Relationship     5233
Attack               515
Name: count, dtype: int64

In [8]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
GEN_MAX_NEW_TOKENS = 3                
TEMPERATURE = 0.0
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}


args_classified = classify_relationships_flan(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_flanT5"].value_counts()

Using MAX_LENGTH = 477

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: to achieve the sdgs the world must both alter its current investment patterns and increase the overall volume of investments.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: the grim reality is that at the midpoint of the 2030 agenda, the sdgs are far off track.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: at the global level, averaging across countries, not a single sdg is currently projected to be met by 2030, with the poorest countries struggling the most.
Label: Support


- Arg1: at the midpoint of the 2030 agenda, all of the sdgs are seriously off track.
-Arg2: 1. increased funding from the multilateral deve

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL,rel_flanT5
9118,The SDG Index relies on inputs from the SDSN n...,2. Universal health coverage,0_125,3_12,NaN,No Relationship,No Relationship
90420,Governments are only now mapping out pathways ...,"For example, globalized trade rules for ‘clean...",4_14,17_30,NaN,Support,No Relationship
530,UN Member States must endorse a deep and overd...,Extreme poverty rates in LICs remain above pre...,0_17,1_20,NaN,No Relationship,No Relationship
105074,The SDGs are not only a public policy framewor...,"Working together with the IMF and the MDBs, th...",8_9,17_44,NaN,No Relationship,Support
32444,"Despite this alarming development, the SDGs ar...",Curtailing the extraction and use of fossil fu...,0_1,13_21,NaN,Support,Support


rel_flanT5
Support            74770
No Relationship    55161
Attack              5584
Name: count, dtype: int64